# 03 — Incremental Load Validation

This notebook validates the incremental extraction flow from FIPEX GitHub Releases.

## Current scope

Implemented and tested so far:

- Resolve FIPEX monthly release
- Select the original unmerged Parquet asset
- Download the full release snapshot temporarily
- Filter the requested reference month
- Persist one monthly Bronze Parquet
- Validate period integrity
- Validate idempotency
- Inspect local Bronze coverage
- List remote releases
- Prepare missing-period / catch-up logic

This notebook intentionally stops at the current project stage.


In [ ]:
from pathlib import Path
import pandas as pd

from fipe_pipeline.extract import (
    extract_month,
    inspect_local_bronze,
    list_available_releases,
)

pd.set_option("display.max_columns", None)

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent

PROJECT_ROOT


## Validate one known published month


In [ ]:
result = extract_month(2026, 9)
result


Expected behavior after the first successful download:

```text
status='already_exists'
rows=51012
destination=.../data/bronze/monthly/fipe_2026_09.parquet
```

The first execution would have returned `status='downloaded'`.


In [ ]:
monthly_path = PROJECT_ROOT / "data" / "bronze" / "monthly" / "fipe_2026_09.parquet"
monthly_path.exists(), monthly_path


In [ ]:
df_2026_09 = pd.read_parquet(monthly_path)
df_2026_09.shape


In [ ]:
df_2026_09[["ano_referencia", "mes_referencia"]].drop_duplicates()


## Inspect local Bronze coverage


In [ ]:
inventory = inspect_local_bronze()
inventory


Expected current local state:

```text
historical watermark = 2026-08
monthly periods      = 2026-09
```


## List FIPEX releases


In [ ]:
releases = list_available_releases()
[(release.period.label, release.tag) for release in releases[-10:]]


## Next implementation checkpoint

The next step is to finish and validate the automatic catch-up flow:

```text
local historical watermark
        +
local monthly inventory
        +
remote FIPEX releases
        ↓
missing periods
        ↓
chronological extraction
```

The pipeline must not silently skip an intermediate month if a newer release exists.
